# 09 — Visual Odometry on TUM fr1/xyz (Document 08 Part 2)

**Author:** Serhii Polishchuk  
**Course:** Computer Vision Engineering for DefenceTech  
**Dataset:** TUM RGB-D `rgbd_dataset_freiburg1_xyz`  
**Pipeline:** OpenCV RGB-D odometry + evo (ATE/RPE)

Steps: Setup → Download → VO run → ATE/RPE → Drift → Degraded (blur) → Compare.

## 1. Setup

In [ ]:
!pip install -q evo opencv-python-headless matplotlib numpy
import cv2, numpy as np, re, shutil
from pathlib import Path
import matplotlib.pyplot as plt
print("OpenCV", cv2.__version__)

## 2. Download TUM fr1/xyz

In [ ]:
ROOT = Path("/content/dz09_out")
ROOT.mkdir(parents=True, exist_ok=True)
TUM = Path("/content/tum")
TUM.mkdir(parents=True, exist_ok=True)
DATA = TUM / "rgbd_dataset_freiburg1_xyz"

if not DATA.exists():
    !wget -q -O /content/tum/fr1_xyz.tgz https://vision.in.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_xyz.tgz
    !tar -xzf /content/tum/fr1_xyz.tgz -C /content/tum
print("DATA:", DATA)
print(sorted([p.name for p in DATA.iterdir()])[:15])

## 3. Associate RGB–Depth and load calibration

TUM Freiburg1 typical intrinsics (fx, fy, cx, cy). Depth scale = 5000.

In [ ]:
def read_list(path):
    rows = []
    for line in Path(path).read_text().splitlines():
        if line.startswith("#") or not line.strip():
            continue
        t, f = line.split()
        rows.append((float(t), f))
    return rows

def associate(a, b, max_diff=0.02):
    """Match timestamps nearest within max_diff seconds."""
    b = list(b)
    pairs = []
    for ta, fa in a:
        best, bj = None, None
        for i, (tb, fb) in enumerate(b):
            d = abs(ta - tb)
            if best is None or d < best:
                best, bj = d, i
        if best is not None and best <= max_diff:
            tb, fb = b.pop(bj)
            pairs.append((ta, fa, tb, fb))
    return pairs

rgb_list = read_list(DATA / "rgb.txt")
dep_list = read_list(DATA / "depth.txt")
pairs = associate(rgb_list, dep_list, max_diff=0.02)
print("associated pairs:", len(pairs))

# Freiburg1 calibration (dataset default)
FX, FY, CX, CY = 517.3, 516.5, 318.6, 255.3
DEPTH_SCALE = 5000.0  # raw depth / scale -> meters
K = np.array([[FX, 0, CX], [0, FY, CY], [0, 0, 1]], dtype=np.float64)
print("K:\n", K)

# save calib artifact
calib_path = ROOT / "calib_fr1.txt"
calib_path.write_text(
    f"fx={FX}\nfy={FY}\ncx={CX}\ncy={CY}\ndepth_scale={DEPTH_SCALE}\n"
    f"dataset=rgbd_dataset_freiburg1_xyz\n"
)
print("saved", calib_path)

## 4. RGB-D Visual Odometry (OpenCV)

Frame-to-frame RGB-D odometry. Trajectory saved in **TUM format**:
`timestamp tx ty tz qx qy qz qw`

In [ ]:
def depth_to_meters(depth_u16):
    d = depth_u16.astype(np.float32) / DEPTH_SCALE
    d[depth_u16 == 0] = 0
    return d

def mat_to_tum_pose(T):
    """4x4 SE3 -> tx ty tz qx qy qz qw"""
    R, t = T[:3, :3], T[:3, 3]
    # rotation matrix to quaternion (x,y,z,w)
    qw = np.sqrt(max(0, 1 + R[0,0] + R[1,1] + R[2,2])) / 2
    qx = np.sign(R[2,1] - R[1,2]) * np.sqrt(max(0, 1 + R[0,0] - R[1,1] - R[2,2])) / 2
    qy = np.sign(R[0,2] - R[2,0]) * np.sqrt(max(0, 1 - R[0,0] + R[1,1] - R[2,2])) / 2
    qz = np.sign(R[1,0] - R[0,1]) * np.sqrt(max(0, 1 - R[0,0] - R[1,1] + R[2,2])) / 2
    return t[0], t[1], t[2], qx, qy, qz, qw

def run_vo(pairs, blur_ksize=0, max_frames=None, tag="baseline"):
    """RGB-D odometry. blur_ksize>0 applies Gaussian blur to RGB (degraded)."""
    odosettings = cv2.rgbd.RgbdOdometry_create()
    # Prefer RgbdOdometry if available; fallback to simple ICP-style via cv2.ppf_match_3d not needed
    use_rgbd = hasattr(cv2, "rgbd") and hasattr(cv2.rgbd, "RgbdOdometry_create")

    poses = []  # list of (t, 4x4)
    T_wc = np.eye(4)
    poses.append((pairs[0][0], T_wc.copy()))

    prev_gray, prev_depth = None, None
    n_ok, n_fail = 0, 0
    limit = len(pairs) if max_frames is None else min(max_frames, len(pairs))

    for i in range(limit):
        ta, fa, tb, fd = pairs[i]
        rgb = cv2.imread(str(DATA / fa), cv2.IMREAD_COLOR)
        depth = cv2.imread(str(DATA / fd), cv2.IMREAD_UNCHANGED)
        if rgb is None or depth is None:
            n_fail += 1
            continue
        if blur_ksize and blur_ksize > 1:
            k = blur_ksize if blur_ksize % 2 == 1 else blur_ksize + 1
            rgb = cv2.GaussianBlur(rgb, (k, k), 0)
        gray = cv2.cvtColor(rgb, cv2.COLOR_BGR2GRAY)
        depth_m = depth_to_meters(depth)

        if prev_gray is None:
            prev_gray, prev_depth = gray, depth_m
            continue

        T_rel = np.eye(4)
        ok = False
        if use_rgbd:
            try:
                odo = cv2.rgbd.RgbdOdometry_create()
                # Rt transforms current to previous in some OpenCV versions; we accumulate carefully
                is_ok, Rt = False, np.eye(4)
                # API: compute(srcImage, srcDepth, srcMask, dstImage, dstDepth, dstMask, Rt, ...)
                is_ok, Rt = odo.compute(prev_gray, prev_depth.astype(np.float32), None,
                                       gray, depth_m.astype(np.float32), None)
                if is_ok and Rt is not None:
                    T_rel = Rt.astype(np.float64)
                    ok = True
            except Exception as e:
                ok = False

        if not ok:
            # Fallback: feature-based PnP with depth
            orb = cv2.ORB_create(2000)
            kp1, des1 = orb.detectAndCompute(prev_gray, None)
            kp2, des2 = orb.detectAndCompute(gray, None)
            if des1 is None or des2 is None or len(kp1) < 8 or len(kp2) < 8:
                n_fail += 1
                prev_gray, prev_depth = gray, depth_m
                poses.append((ta, T_wc.copy()))
                continue
            bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
            matches = sorted(bf.match(des1, des2), key=lambda m: m.distance)[:400]
            pts3d, pts2d = [], []
            for m in matches:
                u1, v1 = kp1[m.queryIdx].pt
                u2, v2 = kp2[m.trainIdx].pt
                z = prev_depth[int(round(v1)), int(round(u1))]
                if z <= 0.1 or z > 8.0:
                    continue
                x = (u1 - CX) * z / FX
                y = (v1 - CY) * z / FY
                pts3d.append([x, y, z])
                pts2d.append([u2, v2])
            if len(pts3d) < 12:
                n_fail += 1
                prev_gray, prev_depth = gray, depth_m
                poses.append((ta, T_wc.copy()))
                continue
            pts3d = np.asarray(pts3d, dtype=np.float64)
            pts2d = np.asarray(pts2d, dtype=np.float64)
            ok_pnp, rvec, tvec, inliers = cv2.solvePnPRansac(
                pts3d, pts2d, K, None, flags=cv2.SOLVEPNP_EPNP, iterationsCount=100,
                reprojectionError=3.0, confidence=0.99
            )
            if not ok_pnp:
                n_fail += 1
                prev_gray, prev_depth = gray, depth_m
                poses.append((ta, T_wc.copy()))
                continue
            R, _ = cv2.Rodrigues(rvec)
            T_rel = np.eye(4)
            T_rel[:3, :3] = R
            T_rel[:3, 3] = tvec.ravel()
            # T_rel: transforms points from prev cam to current? solvePnP maps model(prev 3D) to image(current)
            # so T maps prev -> current; world update T_wc = T_wc @ inv(T_rel) if T is curr from prev
            # Standard: T_rel is pose of world in camera; we use T_wc = T_wc @ inv(T_rel)
            T_rel = np.linalg.inv(T_rel)
            ok = True
            n_ok += 1
        else:
            n_ok += 1

        T_wc = T_wc @ T_rel
        poses.append((ta, T_wc.copy()))
        prev_gray, prev_depth = gray, depth_m
        if (i + 1) % 50 == 0:
            print(f"{tag}: frame {i+1}/{limit} ok={n_ok} fail={n_fail}")

    out = ROOT / f"traj_est_{tag}.txt"
    lines = []
    for t, T in poses:
        tx, ty, tz, qx, qy, qz, qw = mat_to_tum_pose(T)
        lines.append(f"{t:.6f} {tx:.6f} {ty:.6f} {tz:.6f} {qx:.6f} {qy:.6f} {qz:.6f} {qw:.6f}\n")
    out.write_text("".join(lines))
    print(f"{tag}: saved {out} poses={len(poses)} ok={n_ok} fail={n_fail}")
    return out, n_ok, n_fail

print("VO helpers ready")

In [ ]:
# Baseline run (may take several minutes on CPU)
traj_base, ok_b, fail_b = run_vo(pairs, blur_ksize=0, max_frames=None, tag="baseline")
print("baseline ok/fail", ok_b, fail_b)

## 5. Prepare GT and evaluate ATE / RPE with evo

Alignment: **SE(3)** (RGB-D metric scale).  
Time assoc max difference: **0.02 s**.  
Axis convention: TUM (x right, y down, z forward in camera; world as in GT).

In [ ]:
# Copy GT to output
gt_src = DATA / "groundtruth.txt"
gt_dst = ROOT / "groundtruth.txt"
shutil.copy(gt_src, gt_dst)

# Config artifact
cfg = ROOT / "run_config_baseline.yaml"
cfg.write_text(f"""
dataset: rgbd_dataset_freiburg1_xyz
method: opencv_rgbd_orb_pnp_fallback
tag: baseline
blur_ksize: 0
intrinsics: {{fx: {FX}, fy: {FY}, cx: {CX}, cy: {CY}}}
depth_scale: {DEPTH_SCALE}
assoc_max_diff_s: 0.02
eval:
  alignment: SE3
  t_max_diff: 0.02
  pose_relation: ATE / RPE
""".strip() + "\n")
print("config", cfg)

# evo ATE (ape) SE(3)
!evo_ape tum {gt_dst} {traj_base} -as --t_max_diff 0.02 --save_results {ROOT}/ape_baseline.zip -p --plot_mode xy --save_plot {ROOT}/ape_baseline.png

# evo RPE delta 0.5 s and 1.0 s (approx via meters if --delta_unit m not time; use frames proxy)
# For time-based RPE evo supports --delta with unit s in recent evo:
!evo_rpe tum {gt_dst} {traj_base} -as --t_max_diff 0.02 --delta 1 --delta_unit m --save_results {ROOT}/rpe_baseline_1m.zip
!evo_rpe tum {gt_dst} {traj_base} -as --t_max_diff 0.02 --delta 0.5 --delta_unit m --save_results {ROOT}/rpe_baseline_0.5m.zip

print("ATE/RPE done — see plots and zips in", ROOT)

## 6. Degraded scenario: Gaussian blur on RGB

Same segment, controlled blur (ksize=15), seed not needed (deterministic filter).

In [ ]:
traj_deg, ok_d, fail_d = run_vo(pairs, blur_ksize=15, max_frames=None, tag="degraded_blur15")
print("degraded ok/fail", ok_d, fail_d)

(ROOT / "run_config_degraded_blur15.yaml").write_text(f"""
dataset: rgbd_dataset_freiburg1_xyz
method: opencv_rgbd_orb_pnp_fallback
tag: degraded_blur15
blur_ksize: 15
intrinsics: {{fx: {FX}, fy: {FY}, cx: {CX}, cy: {CY}}}
depth_scale: {DEPTH_SCALE}
assoc_max_diff_s: 0.02
eval:
  alignment: SE3
  t_max_diff: 0.02
""".strip()+"\n")

!evo_ape tum {gt_dst} {traj_deg} -as --t_max_diff 0.02 --save_results {ROOT}/ape_degraded.zip -p --plot_mode xy --save_plot {ROOT}/ape_degraded.png
!evo_rpe tum {gt_dst} {traj_deg} -as --t_max_diff 0.02 --delta 1 --delta_unit m --save_results {ROOT}/rpe_degraded_1m.zip
print("degraded eval done")

## 7. Overlay trajectories (xy)

For the report: estimated vs GT.

In [ ]:
def load_tum(path):
    ts, xyz = [], []
    for line in Path(path).read_text().splitlines():
        if line.startswith("#") or not line.strip():
            continue
        p = line.split()
        ts.append(float(p[0]))
        xyz.append([float(p[1]), float(p[2]), float(p[3])])
    return np.array(ts), np.array(xyz)

t_gt, xyz_gt = load_tum(gt_dst)
t_e, xyz_e = load_tum(traj_base)

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.plot(xyz_gt[:, 0], xyz_gt[:, 1], label="GT", linewidth=2)
ax.plot(xyz_e[:, 0], xyz_e[:, 1], label="VO baseline", alpha=0.8)
ax.set_aspect("equal")
ax.legend()
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("fr1/xyz — GT vs VO (before SE3 align in plot)")
fig.tight_layout()
fig.savefig(ROOT / "overlay_xy_baseline.png", dpi=120)
plt.show()
print("saved overlay")

## 8. README command (copy into repo README)

```bash
# One-shot (Colab or local with same paths)
jupyter nbconvert --to notebook --execute 09_VO_TUM_fr1_xyz_Serhii_Polishchuk.ipynb
# Artifacts: dz09_out/traj_est_baseline.txt, calib_fr1.txt, run_config_*.yaml, ape_*.zip
```

Eval parameters for the report:
- **Alignment:** SE(3) (`evo -as` / Umeyama with scale fixed metric)
- **t_max_diff:** 0.02 s
- **Convention:** TUM RGB-D axes / timestamps

After runs, paste ATE RMSE / RPE into Document 08 Part 2.